# Final mushroom preparation contract

**Contribution owner: Muneeb Shakoor - integration, baseline reproduction, AutoML and error analysis (tasks 13, 15, 18, 24).**
Builds on Viviana Pajic's Mushroom exploration/evaluation (tasks 8/11) and the shared
AI-assisted starter maintained by Tomislav Novosel. Codex implemented and executed
this work at Muneeb's request on 8-9 October 2026. Muneeb selected the scope and
preservation requirements; this does not claim independent human authorship or
completed human code review. See [AI assistance](../docs/AI_USE.md) and
[reproduction and decisions](../docs/MUSHROOM_WORKFLOW.md).

Use Python 3.11 with `requirements-pycaret.lock.txt` for the recorded environment.


This notebook transforms the verified teacher file into model-ready development
inputs, without exploratory graphs. EDA evidence remains in notebook 00.
We drop the two noise columns, keep every row, keep nominal categorical codes and
defer fitted imputation/encoding/scaling to each training fold. The original 1,000
test rows are frozen; no test model prediction is computed.


## Load, verify and preserve the existing split

The source checksum and split manifest prevent silently replacing the dataset or moving test rows.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd()
while not (ROOT / 'src/mushrooms.py').exists():
    if ROOT == ROOT.parent:
        raise RuntimeError('Open the notebook inside the repository.')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
import json
import pandas as pd
from mushroom_workflow import REPORTS, PROCESSED
from mushroom_workflow import prepare, candidates, CONTRACT_VERSION
X_dev, y_dev, splits = prepare()
print('Contract:', CONTRACT_VERSION)
print('Development:', X_dev.shape, '| reserved test:', len(splits['test']))
print('Development classes:', y_dev.value_counts().sort_index().to_dict(), '(0=edible, 1=poisonous)')
assert not set(X_dev.index) & set(splits['test'])
display(pd.DataFrame({'dtype': X_dev.dtypes.astype(str), 'missing': X_dev.isna().sum()}))


Contract: mushroom-development-v1
Development: (4000, 10) | reserved test: 1000
Development classes: {0: 2485, 1: 1515} (0=edible, 1=poisonous)


,dtype,missing
cap-diameter,float64,1613
stem-height,float64,811
stem-width,float64,321
spore-print-color,object,3804
gill-color,object,786
habitat,object,793
season,object,786
ring-type,object,457
cap-shape,object,322
stem-surface,object,2691


## Model-ready raw features, not a globally fitted matrix

`development.csv` preserves source-row IDs and the target. Its ten raw feature
columns can be passed directly into each complete model pipeline. Numeric medians,
category vocabularies and LR scaling must be learned separately inside each fold.
The example below fits only one training fold and transforms its validation fold;
those matrices are an illustration and are not reused across CV folds.


In [2]:
from mushrooms import cv_splitter
train, validation = next(cv_splitter().split(X_dev, y_dev))
prepare_fold = candidates(X_dev)['random_forest'].named_steps['prepare']
prepared_train = prepare_fold.fit_transform(X_dev.iloc[train])
prepared_validation = prepare_fold.transform(X_dev.iloc[validation])
print('Example fold:', prepared_train.shape, prepared_validation.shape)
print('Generated model-ready raw input:', (PROCESSED / 'development.csv').relative_to(ROOT))
print('No numeric missing indicators; missing categories are encoded separately.')


Example fold: (3200, 64) (800, 64)
Generated model-ready raw input: data\processed\mushrooms\development.csv
No numeric missing indicators; missing categories are encoded separately.


## Retained-feature collisions

This is a data-integrity audit, not test performance evaluation. At the user's
request, label agreement is checked for the collision groups, including the group
crossing development/test. No rows or test memberships are changed. Equal retained
inputs do not prove repeated source observations; the original source has no ID.
The model notebook measures sensitivity using development folds only.


In [3]:
collisions = json.loads((REPORTS / 'collisions.json').read_text())
display(pd.DataFrame([{
    'source_rows': [r['source_row'] for r in g['rows']],
    'partitions': [r['partition'] for r in g['rows']],
    'labels_agree': g['labels_agree'], 'label_counts': g['label_counts'],
    'missingness_identical': g['missingness_identical'],
    'missing_features': g['missing_features'],
} for g in collisions['groups']]))
print(collisions['policy'])


,source_rows,partitions,labels_agree,label_counts,missingness_identical,missing_features
0,"[232, 3164, 4244]","[train, test, test]",True,{'p': 3},True,"[cap-diameter, spore-print-color]"
1,"[316, 3608]","[train, train]",True,{'e': 2},True,"[cap-diameter, stem-height, stem-width, spore-..."
2,"[580, 4082]","[train, validation]",True,{'p': 2},True,"[cap-diameter, spore-print-color, gill-color]"
3,"[1770, 3851]","[validation, train]",True,{'e': 2},True,"[cap-diameter, stem-height, stem-width, spore-..."


Keep all rows and the original test membership. Measure development sensitivity by excluding matching training predictors within each CV fold only.


## Preparation decision

Retain all observations and original numeric values. Use the EDA-supported ten
predictors, median numeric imputation without extra indicators, and categorical
`__MISSING__` plus one-hot encoding. Only LR scales numeric values. Unseen category
codes are handled by the encoder. These are modelling choices, not claims that
missingness is MCAR, that all outliers are verified, or that zeros have a proven
biological meaning. Continue to notebook 02; do not fit preprocessing globally.
